In [11]:
"""
Climate-Induced Migration Pressure Modeling — India (District-Level)
Step 1: Data Loading & Inspection
"""
 
import pandas as pd
 
# ─────────────────────────────────────────────
# File paths
# ─────────────────────────────────────────────
CENSUS_PATH   = "C:/Users/asus/OneDrive/Documents/climate-migration-prediction/data/raw/census.xlsx"
RAINFALL_PATH = "C:/Users/asus/OneDrive/Documents/climate-migration-prediction/data/raw/rainfall.csv"
MPI_PATH      = "C:/Users/asus/OneDrive/Documents/climate-migration-prediction/data/raw/mpi.xlsx"
AGRI_PATH     = "C:/Users/asus/OneDrive/Documents/climate-migration-prediction/data/raw/agriculture.csv"
 
DIVIDER = "=" * 70

In [12]:
# ─────────────────────────────────────────────
# Utility helpers
# ─────────────────────────────────────────────
 
def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Lowercase column names and replace spaces with underscores."""
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(r"[\s\-]+", "_", regex=True)
        .str.replace(r"[^\w]", "", regex=True)
    )
    return df
 
 
def inspect_dataframe(name: str, df: pd.DataFrame) -> None:
    """Print shape, columns, dtypes, head, missing values, duplicates, and stats."""
    print(f"\n{DIVIDER}")
    print(f"  DATASET: {name}")
    print(DIVIDER)
 
    # Shape
    print(f"\n[Shape]\n  Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}")
 
    # Column names
    print(f"\n[Columns ({df.shape[1]})]")
    for col in df.columns:
        print(f"  • {col}")
 
    # Data types
    print(f"\n[Data Types]")
    print(df.dtypes.to_string())
 
    # First 5 rows
    print(f"\n[First 5 Rows]")
    print(df.head().to_string(index=False))
 
    # Missing values
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    missing_summary = pd.DataFrame({
        "missing_count": missing,
        "missing_pct":   missing_pct,
    })
    missing_summary = missing_summary[missing_summary["missing_count"] > 0]
 
    print(f"\n[Missing Values]")
    if missing_summary.empty:
        print("  ✓ No missing values found.")
    else:
        print(missing_summary.to_string())
 
    # Duplicate rows
    n_dupes = df.duplicated().sum()
    print(f"\n[Duplicate Rows]")
    if n_dupes == 0:
        print("  ✓ No duplicate rows found.")
    else:
        print(f"  ⚠ {n_dupes:,} duplicate row(s) detected.")
 
    # Summary statistics — numerical columns only
    num_df = df.select_dtypes(include="number")
    if not num_df.empty:
        print(f"\n[Summary Statistics — Numerical Columns]")
        print(num_df.describe().round(4).to_string())
    else:
        print(f"\n[Summary Statistics]\n  No numerical columns found.")
 

In [13]:
# ─────────────────────────────────────────────
# Loaders
# ─────────────────────────────────────────────
 
def load_census(path: str) -> pd.DataFrame:
    df = pd.read_excel(path, dtype=str)          # keep codes as strings
    df = standardize_columns(df)
    # Cast clearly numeric columns back to numeric
    numeric_cols = ["tot_p", "p_lit", "margwork_p", "p_sc", "p_st"]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df
 
 
def load_rainfall(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    df = standardize_columns(df)
    return df
 
 
def load_mpi(path: str) -> pd.DataFrame:
    """
    The MPI Excel file has 4 rows of title/description text before the
    multi-row merged header block.  Row index 4 (0-based) contains the
    top-level header; row 5 contains the sub-level labels we care about.
    Data begins at row index 9.  We build a clean flat header manually.
    """
    raw = pd.read_excel(path, header=None)
 
    # The cleanest single-row header is at row 5 (0-based); fill forward
    # the State/District/Country labels from row 4.
    header_row4 = raw.iloc[4].ffill().tolist()
    header_row5 = raw.iloc[5].tolist()
 
    flat_header = []
    for h4, h5 in zip(header_row4, header_row5):
        h4 = str(h4).strip() if pd.notna(h4) else ""
        h5 = str(h5).strip() if pd.notna(h5) else ""
        if h5 and h5 != "nan" and h5 not in (h4,):
            flat_header.append(f"{h4} — {h5}".strip(" — "))
        else:
            flat_header.append(h4)
 
    # Data rows start at index 9
    data = raw.iloc[9:].reset_index(drop=True)
    data.columns = flat_header
 
    # Drop fully-empty columns (artefacts of merged cells)
    data = data.dropna(axis=1, how="all")
    data = standardize_columns(data)
    return data
 
 
def load_agriculture(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    df = standardize_columns(df)
    return df

In [14]:
# ─────────────────────────────────────────────
# Main
# ─────────────────────────────────────────────
 
def main():
    print("\nLoading datasets …")
 
    df_census   = load_census(CENSUS_PATH)
    df_rainfall = load_rainfall(RAINFALL_PATH)
    df_mpi      = load_mpi(MPI_PATH)
    df_agri     = load_agriculture(AGRI_PATH)
 
    print("All datasets loaded successfully.\n")
 
    datasets = {
        "Census PCA (2011)":             df_census,
        "Rainfall (1901–2017)":          df_rainfall,
        "MPI (2015–16)":                 df_mpi,
        "Agriculture (2000–2011)":       df_agri,
    }
 
    for name, df in datasets.items():
        inspect_dataframe(name, df)
 
    print(f"\n{DIVIDER}")
    print("  END OF INSPECTION  —  No preprocessing performed.")
    print(DIVIDER)
 
 
if __name__ == "__main__":
    main()


Loading datasets …
All datasets loaded successfully.


  DATASET: Census PCA (2011)

[Shape]
  Rows: 2,028  |  Columns: 94

[Columns (94)]
  • state
  • district
  • subdistt
  • townvillage
  • ward
  • eb
  • level
  • name
  • tru
  • no_hh
  • tot_p
  • tot_m
  • tot_f
  • p_06
  • m_06
  • f_06
  • p_sc
  • m_sc
  • f_sc
  • p_st
  • m_st
  • f_st
  • p_lit
  • m_lit
  • f_lit
  • p_ill
  • m_ill
  • f_ill
  • tot_work_p
  • tot_work_m
  • tot_work_f
  • mainwork_p
  • mainwork_m
  • mainwork_f
  • main_cl_p
  • main_cl_m
  • main_cl_f
  • main_al_p
  • main_al_m
  • main_al_f
  • main_hh_p
  • main_hh_m
  • main_hh_f
  • main_ot_p
  • main_ot_m
  • main_ot_f
  • margwork_p
  • margwork_m
  • margwork_f
  • marg_cl_p
  • marg_cl_m
  • marg_cl_f
  • marg_al_p
  • marg_al_m
  • marg_al_f
  • marg_hh_p
  • marg_hh_m
  • marg_hh_f
  • marg_ot_p
  • marg_ot_m
  • marg_ot_f
  • margwork_3_6_p
  • margwork_3_6_m
  • margwork_3_6_f
  • marg_cl_3_6_p
  • marg_cl_3_6_m
  • marg_cl_3_6_f
  